In [1]:
"""
Stage 3: Seasonality Detection
Comprehensive check for daily, weekly, and yearly seasonality patterns
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
from scipy import signal


In [2]:
# loading data
from ucimlrepo import fetch_ucirepo 
seoul_bike_sharing_demand = fetch_ucirepo(id=560) 
  
# data (as pandas dataframes) 
X = seoul_bike_sharing_demand.data.features 
y = seoul_bike_sharing_demand.data.targets 
  
# Combine for easier handling
df_concat = pd.concat([X, y], axis=1)

# Create datetime
df_concat['DateTime'] = pd.to_datetime(df_concat['Date'], format='%d/%m/%Y') + pd.to_timedelta(df_concat['Hour'], unit='h')
df = df_concat.copy()
df.head(3)

,Date,Rented Bike Count,Hour,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Seasons,Holiday,Functioning Day,DateTime
0,1/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 00:00:00
1,1/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 01:00:00
2,1/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 02:00:00


In [3]:
# Prepare data
df_ts = df.set_index('DateTime').sort_index()
target = 'Rented Bike Count'
y = df_ts[target]

print("="*70)
print("SEASONALITY DETECTION")
print("="*70)


SEASONALITY DETECTION


In [4]:

# ============================================================================
# 1. ACF PLOT - Visual inspection of periodic patterns
# ============================================================================
print("\n[1] ACF ANALYSIS")
print("-"*70)

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# ACF with different lag ranges to see different seasonalities
lag_configs = [
    (72, "ACF: First 3 days (72 hours) - Check daily pattern"),
    (336, "ACF: First 2 weeks (336 hours) - Check weekly pattern"),
    (1000, "ACF: First ~42 days (1000 hours) - Check longer patterns")
]

for idx, (lags, title) in enumerate(lag_configs):
    plot_acf(y, lags=lags, ax=axes[idx], alpha=0.05)
    axes[idx].set_title(title)
    axes[idx].set_xlabel('Lag (hours)')
    axes[idx].set_ylabel('Autocorrelation')

plt.tight_layout()
plt.savefig('img/seasonality_acf_seoul.png', dpi=300, bbox_inches='tight')
print("ACF plots saved as 'seasonality_acf_seoul.png'")
plt.close()



[1] ACF ANALYSIS
----------------------------------------------------------------------
ACF plots saved as 'seasonality_acf_seoul.png'


In [5]:

# ============================================================================
# 2. AUTOCORRELATION AT SPECIFIC LAGS
# ============================================================================
print("\n[2] AUTOCORRELATION AT KEY LAGS")
print("-"*70)

# Calculate autocorrelation for specific lags
specific_lags = {
    'Daily (24h)': 24,
    'Weekly (168h)': 168,
    'Bi-weekly (336h)': 336,
    'Monthly (~730h)': 730,
    'Yearly (8760h)': 8760
}

acf_values = acf(y, nlags=max(specific_lags.values()), fft=True)

print("\nAutocorrelation values at key lags:")
for name, lag in specific_lags.items():
    if lag < len(acf_values):
        print(f"  {name:20s} lag={lag:5d}  ACF={acf_values[lag]:.4f}")
    else:
        print(f"  {name:20s} lag={lag:5d}  [Not enough data]")



[2] AUTOCORRELATION AT KEY LAGS
----------------------------------------------------------------------

Autocorrelation values at key lags:
  Daily (24h)          lag=   24  ACF=0.6813
  Weekly (168h)        lag=  168  ACF=0.6559
  Bi-weekly (336h)     lag=  336  ACF=0.6346
  Monthly (~730h)      lag=  730  ACF=0.0347
  Yearly (8760h)       lag= 8760  [Not enough data]


In [6]:

# ============================================================================
# 3. VISUAL AGGREGATION
# ============================================================================
print("\n[3] VISUAL AGGREGATION - Average patterns")
print("-"*70)

# Extract time components
df_agg = df.copy()
df_agg['Hour'] = df_agg['DateTime'].dt.hour
df_agg['DayOfWeek'] = df_agg['DateTime'].dt.dayofweek  # 0=Monday, 6=Sunday
df_agg['Month'] = df_agg['DateTime'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3a. Daily pattern: Average by hour
hourly_avg = df_agg.groupby('Hour')[target].agg(['mean', 'std'])
axes[0].plot(hourly_avg.index, hourly_avg['mean'], marker='o', linewidth=2)
axes[0].fill_between(hourly_avg.index, 
                      hourly_avg['mean'] - hourly_avg['std'],
                      hourly_avg['mean'] + hourly_avg['std'],
                      alpha=0.3)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Bike Count')
axes[0].set_title('Daily Pattern: Average by Hour')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(0, 24, 2))

# 3b. Weekly pattern: Average by day of week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekly_avg = df_agg.groupby('DayOfWeek')[target].agg(['mean', 'std'])
axes[1].bar(weekly_avg.index, weekly_avg['mean'], yerr=weekly_avg['std'], 
            capsize=5, alpha=0.7)
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Average Bike Count')
axes[1].set_title('Weekly Pattern: Average by Day')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_names)
axes[1].grid(True, alpha=0.3, axis='y')

# 3c. Yearly pattern: Average by month
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg = df_agg.groupby('Month')[target].agg(['mean', 'std'])
axes[2].plot(monthly_avg.index, monthly_avg['mean'], marker='o', linewidth=2)
axes[2].fill_between(monthly_avg.index,
                      monthly_avg['mean'] - monthly_avg['std'],
                      monthly_avg['mean'] + monthly_avg['std'],
                      alpha=0.3)
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Average Bike Count')
axes[2].set_title('Yearly Pattern: Average by Month')
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(month_names, rotation=45)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('img/seasonality_visual_aggregation_seoul.png', dpi=300, bbox_inches='tight')
print("Visual aggregation saved as 'seasonality_visual_aggregation_seoul.png'")
plt.close()

# Print statistics
print("\nPattern strength (coefficient of variation):")
print(f"  Hourly:  CV = {hourly_avg['mean'].std() / hourly_avg['mean'].mean():.4f}")
print(f"  Weekly:  CV = {weekly_avg['mean'].std() / weekly_avg['mean'].mean():.4f}")
print(f"  Monthly: CV = {monthly_avg['mean'].std() / monthly_avg['mean'].mean():.4f}")



[3] VISUAL AGGREGATION - Average patterns
----------------------------------------------------------------------
Visual aggregation saved as 'seasonality_visual_aggregation_seoul.png'

Pattern strength (coefficient of variation):
  Hourly:  CV = 0.5053
  Weekly:  CV = 0.0596
  Monthly: CV = 0.4846


In [7]:

# ============================================================================
# 4. PERIODOGRAM - Frequency domain analysis
# ============================================================================
print("\n[4] PERIODOGRAM - Dominant frequency identification")
print("-"*70)

# Calculate periodogram
frequencies, power = signal.periodogram(y.values, fs=1.0)  # fs=1 for hourly data


# Find dominant frequencies (top 5 peaks)
peak_indices = signal.find_peaks(power, height=np.percentile(power, 95))[0]
peak_freqs = frequencies[peak_indices]
peak_powers = power[peak_indices]




[4] PERIODOGRAM - Dominant frequency identification
----------------------------------------------------------------------


In [8]:

# Sort by power
sorted_indices = np.argsort(peak_powers)[::-1][:5]
dominant_freqs = peak_freqs[sorted_indices]
dominant_powers = peak_powers[sorted_indices]


In [9]:


print("\nTop 5 dominant frequencies:")
for i, (freq, pwr) in enumerate(zip(dominant_freqs, dominant_powers), 1):
    if freq > 0:
        period = 1 / freq
        print(f"  {i}. Frequency: {freq:.6f} -> Period: {period:.2f} hours ({period/24:.2f} days)")
    else:
        print(f"  {i}. Frequency: {freq:.6f} (DC component)")



Top 5 dominant frequencies:
  1. Frequency: 0.000114 -> Period: 8760.00 hours (365.00 days)
  2. Frequency: 0.041667 -> Period: 24.00 hours (1.00 days)
  3. Frequency: 0.083333 -> Period: 12.00 hours (0.50 days)
  4. Frequency: 0.208333 -> Period: 4.80 hours (0.20 days)
  5. Frequency: 0.125000 -> Period: 8.00 hours (0.33 days)


In [10]:

# Plot periodogram
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Full periodogram
axes[0].semilogy(frequencies[1:], power[1:])  # Skip DC component
axes[0].set_xlabel('Frequency (cycles per hour)')
axes[0].set_ylabel('Power Spectral Density')
axes[0].set_title('Periodogram: Full spectrum')
axes[0].grid(True, alpha=0.3)

# Zoomed to show low frequencies (long periods)
max_freq = 0.01  # Show up to 100 hour periods
mask = frequencies < max_freq
axes[1].semilogy(frequencies[mask], power[mask])
axes[1].set_xlabel('Frequency (cycles per hour)')
axes[1].set_ylabel('Power Spectral Density')
axes[1].set_title('Periodogram: Low frequencies (long periods)')
axes[1].grid(True, alpha=0.3)

# Mark expected seasonalities
expected_periods = [24, 168]  # daily, weekly
for period in expected_periods:
    freq = 1 / period
    if freq < max_freq:
        axes[1].axvline(freq, color='red', linestyle='--', alpha=0.5, 
                       label=f'{period}h period')
axes[1].legend()

plt.tight_layout()
plt.savefig('img/seasonality_periodogram_seoul.png', dpi=300, bbox_inches='tight')
print("Periodogram saved as 'seasonality_periodogram_seoul.png'")
plt.close()


Periodogram saved as 'seasonality_periodogram_seoul.png'


In [11]:

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*70)
print("SEASONALITY SUMMARY")
print("="*70)

print("\n1. ACF PLOT FINDINGS:")
print("   - Clear periodic spikes every 24 hours (strong daily pattern)")
print("   - Weekly envelope visible (combined daily + weekly structure)")
print("   - Pattern persists throughout entire time range")

print("\n2. AUTOCORRELATION AT KEY LAGS:")
print("   - Daily (24h): 0.68 - Strong daily seasonality")
print("   - Weekly (168h): 0.66 - Strong weekly pattern")
print("   - Monthly (730h): 0.03 - No monthly periodicity")

print("\n3. VISUAL AGGREGATION:")
print("   - Daily: Bimodal pattern with morning and evening peaks (commute behavior)")
print("   - Weekly: Similar daily averages but different intra-day shapes")
print("   - Yearly: Strong seasonal curve (peak June ~1250, low Dec-Jan ~200)")

print("\n4. PERIODOGRAM:")
print("   - Dominant frequency #1: 365 days (yearly seasonal cycle)")
print("   - Dominant frequency #2: 24 hours (daily cycle)")
print("   - Additional harmonics: 12h, 8h, 4.8h (sub-daily patterns)")

print("\n" + "-"*70)
print("SEEMINGLY CONTRADICTORY RESULTS:")
print("-"*70)
print("Weekly pattern: High ACF but low CV")
print("  -> Weekly pattern affects SHAPE of daily curve, not total volume")
print("  -> Weekday vs weekend have different peak timing, similar totals")
print("\nYearly pattern: Low ACF at 730h but high monthly CV and #1 periodogram")
print("  -> Gradual seasonal trend (365 days), not monthly periodicity")
print("  -> Weather-driven: summer high, winter low")

print("\n" + "="*70)
print("IDENTIFIED SEASONALITIES:")
print("="*70)
print("  1. Daily (24h) - STRONG")
print("  2. Weekly (168h) - MODERATE (shape modulation)")
print("  3. Yearly (365 days) - STRONG (gradual seasonal trend)")
print("\nNext step: Decompose using identified periods")


SEASONALITY SUMMARY

1. ACF PLOT FINDINGS:
   - Clear periodic spikes every 24 hours (strong daily pattern)
   - Weekly envelope visible (combined daily + weekly structure)
   - Pattern persists throughout entire time range

2. AUTOCORRELATION AT KEY LAGS:
   - Daily (24h): 0.68 - Strong daily seasonality
   - Weekly (168h): 0.66 - Strong weekly pattern
   - Monthly (730h): 0.03 - No monthly periodicity

3. VISUAL AGGREGATION:
   - Daily: Bimodal pattern with morning and evening peaks (commute behavior)
   - Weekly: Similar daily averages but different intra-day shapes
   - Yearly: Strong seasonal curve (peak June ~1250, low Dec-Jan ~200)

4. PERIODOGRAM:
   - Dominant frequency #1: 365 days (yearly seasonal cycle)
   - Dominant frequency #2: 24 hours (daily cycle)
   - Additional harmonics: 12h, 8h, 4.8h (sub-daily patterns)

----------------------------------------------------------------------
SEEMINGLY CONTRADICTORY RESULTS:
--------------------------------------------------------

In [1]:
8465+295

8760

In [ ]:
print(df.columns)